# LoRA fine-tuning quickstart

Fine-tune **Qwen3-0.6B** with LoRA on the sample *recruiter assistant* dataset, then chat with it and compare it against the base model.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (free tier is enough). The whole notebook takes about 15 minutes.

Repo: https://github.com/ThomasBaileyUK/llm-lora-finetuning

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install

In [ ]:
!git clone -q https://github.com/ThomasBaileyUK/llm-lora-finetuning.git
%cd llm-lora-finetuning
!pip install -q -e ".[qlora]"

## 2. (Optional) smoke test
10 steps on a 135M-parameter model. Proves everything is wired up in about a minute.

In [ ]:
!python -m lora_ft.train -c configs/smoke_test.yaml

## 3. Train the LoRA adapter
Only about 1–2% of the weights are trained. On a T4 the app automatically uses fp16, because T4 GPUs don't support bf16.

In [ ]:
!python -m lora_ft.train -c configs/qwen3_0.6b_lora.yaml

In [ ]:
import json
from IPython.display import Image, display
s = json.load(open("outputs/qwen3-0.6b-lora/summary.json"))
print(f"Trainable params: {s['trainable_params']:,} of {s['total_params']:,} ({100*s['trainable_params']/s['total_params']:.2f}%)")
print(f"Training time: {s['minutes']} min on {s['device']} ({s['dtype']})")
print({k: round(v, 4) for k, v in s['metrics'].items() if isinstance(v, (int, float))})
display(Image("outputs/qwen3-0.6b-lora/loss.png"))

## 4. Base model vs fine-tuned

In [ ]:
!python -m lora_ft.chat --adapter outputs/qwen3-0.6b-lora/adapter --compare --temperature 0 --prompt "What is a forward deployed engineer?"

In [ ]:
!python -m lora_ft.chat --adapter outputs/qwen3-0.6b-lora/adapter --compare --temperature 0 --prompt "Write a 2-line outreach to a data engineer for an AI startup in Berlin."

## 5. Evaluation report
Perplexity on the held-out answers, plus side-by-side generations, written to `outputs/qwen3-0.6b-lora/eval_report.md`.

In [ ]:
!python -m lora_ft.evaluate --adapter outputs/qwen3-0.6b-lora/adapter --data data/recruiter_assistant/eval.jsonl
from IPython.display import Markdown
Markdown(open("outputs/qwen3-0.6b-lora/eval_report.md").read())

## 6. (Optional) Share on Hugging Face
Create a **write** token at https://huggingface.co/settings/tokens, then run the next cell.

In [ ]:
from huggingface_hub import HfApi, login

login()  # paste a token with write access
repo_id = "YOUR-USERNAME/qwen3-0.6b-recruiter-lora"
api = HfApi()
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path="outputs/qwen3-0.6b-lora/adapter", repo_id=repo_id)
print(f"https://huggingface.co/{repo_id}")

## Next steps
- Try QLoRA on a 4B model: `!python -m lora_ft.train -c configs/qwen3_4b_qlora.yaml`
- Train on your own data: upload a `.jsonl` file and run with `--set data.train=my_data.jsonl data.eval=null data.eval_fraction=0.1`
- Merge the adapter into a standalone model: `!python -m lora_ft.merge --adapter outputs/qwen3-0.6b-lora/adapter --out merged`